"""
api_extraction.py
=================
Módulo de extracción de datos de la API pública de ClinicalTrials.gov v2.
No requiere API key. Límite: ~50 requests/min por IP.

Uso:
    from src.api_extraction import fetch_trials, fetch_all_trials, save_raw_data
    df = fetch_all_trials(conditions=["cancer", "diabetes"], max_studies=1000)
    save_raw_data(df, "trials_raw.csv")
"""

In [18]:
### Instalación de librerías Instalamos las librerías necesarias para el proyecto
# usando pip.Esto solo se necesita ejecutar una vez.
import subprocess
subprocess.run(["python", "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn", "requests"])


CompletedProcess(args=['python', '-m', 'pip', 'install', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'requests'], returncode=1)

In [20]:
# Importación de librerías. Cargamos las librerías que usaremos a lo largo del proyecto: 

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime


In [6]:

# — Configuración base
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
RAW_DATA_DIR = os.path.join(os.getcwd(), "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)
print(f"✅ Directorio raw: {RAW_DATA_DIR}")


✅ Directorio raw: c:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab\data\raw


In [21]:
#Funcion que realiza la consulta a la API de ClinicalTrials.gov con los parámetros especificados y devuelve los resultados en formato JSON.
def fetch_page(condition=None, page_size=1000, page_token=None):
    params = {"format": "json", "pageSize": page_size, "countTotal": "true"}
    if condition:
        params["query.cond"] = condition
    if page_token:
        params["pageToken"] = page_token
    r = requests.get(BASE_URL, params=params, timeout=30)
    return r.json() if r.status_code == 200 else {}

print("✅ fetch_page OK")

✅ fetch_page OK


In [22]:
# Función parse_study. Función que extrae los campos relevantes de cada estudio y los devuelve en un diccionario plano.
def parse_study(study):
    p = study.get("protocolSection", {})
    id_ = p.get("identificationModule", {})
    st = p.get("statusModule", {})
    d = p.get("designModule", {})
    sp = p.get("sponsorCollaboratorsModule", {})
    locs = p.get("contactsLocationsModule", {}).get("locations", [])
    conds = p.get("conditionsModule", {}).get("conditions", [])
    return {
        "nct_id": id_.get("nctId", ""),
        "title": id_.get("briefTitle", ""),
        "status": st.get("overallStatus", ""),
        "phase": d.get("phases", [None])[0] if d.get("phases") else None,
        "start_date": st.get("startDateStruct", {}).get("date", ""),
        "enrollment": d.get("enrollmentInfo", {}).get("count", None),
        "sponsor": sp.get("leadSponsor", {}).get("name", ""),
        "sponsor_class": sp.get("leadSponsor", {}).get("class", ""),
        "conditions": "; ".join(conds[:3]),
        "countries": "; ".join({l.get("country","") for l in locs if l.get("country")}),
    }

print("✅ parse_study OK")

✅ parse_study OK


In [23]:
# Función fetch_all. Función que itera sobre una lista de condiciones, llama a fetch_page para cada una y acumula los resultados hasta alcanzar un límite máximo de estudios.
def fetch_all(conditions, max_studies=5000):
    records = []
    for term in conditions:
        print(f"Extrayendo: {term}...")
        page_token = None
        while len(records) < max_studies:
            data = fetch_page(condition=term, page_token=page_token)
            studies = data.get("studies", [])
            if not studies:
                break
            for s in studies:
                records.append(parse_study(s))
            page_token = data.get("nextPageToken")
            if not page_token:
                break
            time.sleep(1.2)
        print(f"  Total acumulado: {len(records)}")
    return pd.DataFrame(records)

print("✅ fetch_all OK")

✅ fetch_all OK


In [ ]:
# Extracción de datos. Definimos las condiciones de interés y llamamos a fetch_all para obtener el dataset completo.
CONDITIONS = ["cancer", "diabetes", "cardiovascular", "rare disease"]

df_raw = fetch_all(CONDITIONS, max_studies=4000)
print(f"\nDataset: {df_raw.shape}")
df_raw.head()

Extrayendo: cancer...
  Total acumulado: 4000
Extrayendo: diabetes...
  Total acumulado: 4000
Extrayendo: cardiovascular...
  Total acumulado: 4000
Extrayendo: rare disease...
  Total acumulado: 4000

Dataset: (4000, 10)


,nct_id,title,status,phase,start_date,enrollment,sponsor,sponsor_class,conditions,countries
0,NCT06423690,First in Human Study for the Assessment of Saf...,RECRUITING,NA,2024-11-01,15.0,Snipe Medical,INDUSTRY,Lung Cancer,United Kingdom; Spain; Israel
1,NCT00257790,The Tobramycin Study,COMPLETED,PHASE4,2001-09,210.0,Oslo University Hospital,OTHER,Neutropenia; Fever; Cancer,
2,NCT00369590,VEGF Trap in Treating Patients With Recurrent ...,COMPLETED,PHASE2,2006-08,58.0,National Cancer Institute (NCI),NIH,Adult Anaplastic Astrocytoma; Adult Anaplastic...,United States
3,NCT00513097,Curbing Tobacco Use in Suburban and Rural Schools,COMPLETED,PHASE1,2006-07-25,1289.0,M.D. Anderson Cancer Center,OTHER,Tobacco Use Cessation,United States
4,NCT04501497,Prospective Multicenter Observational Study of...,COMPLETED,None,2020-08-21,1221.0,Chugai Pharmaceutical,INDUSTRY,Non-small Cell Lung Cancer; Extensive Disease ...,Japan


In [24]:
filepath = os.path.join(RAW_DATA_DIR, "trials_raw.csv")
df_raw.to_csv(filepath, index=False, encoding="utf-8")
print(f"✅ Datos guardados: {filepath}")
print(f"Shape: {df_raw.shape}")

✅ Datos guardados: c:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab\data\raw\trials_raw.csv
Shape: (4000, 10)


In [26]:
# Revisamos las primeras filas, tipos de datos, valores nulos y estadísticas básicas del dataset para entender su estructura y calidad.
print("=== PRIMERAS FILAS ===")
print(df_raw.head())
print("\n=== TIPOS DE DATOS ===")
print(df_raw.dtypes)
print("\n=== VALORES NULOS ===")
print(df_raw.isnull().sum())
print("\n=== ESTADÍSTICAS ===")
print(df_raw.describe())

=== PRIMERAS FILAS ===
        nct_id                                              title      status  \
0  NCT06423690  First in Human Study for the Assessment of Saf...  RECRUITING   
1  NCT00257790                               The Tobramycin Study   COMPLETED   
2  NCT00369590  VEGF Trap in Treating Patients With Recurrent ...   COMPLETED   
3  NCT00513097  Curbing Tobacco Use in Suburban and Rural Schools   COMPLETED   
4  NCT04501497  Prospective Multicenter Observational Study of...   COMPLETED   

    phase  start_date  enrollment                          sponsor  \
0      NA  2024-11-01        15.0                    Snipe Medical   
1  PHASE4     2001-09       210.0         Oslo University Hospital   
2  PHASE2     2006-08        58.0  National Cancer Institute (NCI)   
3  PHASE1  2006-07-25      1289.0      M.D. Anderson Cancer Center   
4    None  2020-08-21      1221.0            Chugai Pharmaceutical   

  sponsor_class                                         conditions  \